# 02 — Treino LoRA local + monitoramento

**OBRIGATÓRIO:** Select Kernel → `/Users/wolfx/Documents/Dev/Celx/.venv/bin/python`

Se aparecer `CommandLineTools`, o kernel está ERRADO.

**Watch (outro Terminal):**
```bash
source .venv/bin/activate
python scripts/watch_training.py
```


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

EXPECTED = Path('/Users/wolfx/Documents/Dev/Celx/.venv/bin/python').resolve()
current = Path(sys.executable).resolve()
print("Python do kernel:", current)

if current != EXPECTED and ".venv" not in str(current):
    raise RuntimeError(
        "Kernel ERRADO (Python do sistema).\n"
        "Cursor: Select Kernel (canto do notebook)\n"
        f"Escolha: {EXPECTED}\n"
        f"Atual: {current}\n"
        "Depois: Restart Kernel e rode de novo."
    )

ROOT = Path('/Users/wolfx/Documents/Dev/Celx').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["PYTHONPATH"] = str(ROOT)
print("Kernel OK (.venv)")


## Localizar o projeto


In [ ]:
here = Path.cwd().resolve()
repo_dir = None
for candidate in [here, *here.parents]:
    if (candidate / 'configs' / 'train_local.yaml').exists():
        repo_dir = candidate
        break
assert repo_dir is not None, 'Abra o notebook a partir da raiz do Celx.'
os.environ['PYTHONPATH'] = str(repo_dir)
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
print('Projeto:', repo_dir)


In [ ]:
# Instala deps SÓ no .venv (nunca no Python da Apple)
pkgs = [
    "transformers>=4.51", "accelerate>=1.0", "datasets>=3.0",
    "PyYAML>=6.0", "pandas>=2.0", "peft>=0.14", "trl>=0.15", "torch",
]
print("Instalando no .venv (pode demorar)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
# NÃO usa pip install -e . — evita Permission denied
import legacy_doc
print("legacy_doc OK:", legacy_doc.__version__)
print("Deps OK")


## Dataset SFT

Se ainda não houver curadoria, gera um **smoke** mínimo (só para validar o pipeline local).


In [ ]:
sft_dir = repo_dir / 'dataset/processed/sft'
train_dir = sft_dir / 'train'
validation_dir = sft_dir / 'validation'

if not train_dir.exists() or not validation_dir.exists():
    print('SFT ausente — gerando smoke local...')
    subprocess.run(
        [sys.executable, 'scripts/prepare_smoke_sft.py', '--config', 'configs/train_local.yaml'],
        check=True,
        cwd=repo_dir,
    )
else:
    print('SFT encontrado:', sft_dir)

from datasets import load_from_disk
print('Train:', len(load_from_disk(str(train_dir))))
print('Validation:', len(load_from_disk(str(validation_dir))))


## Monitoramento (TensorBoard)

No Terminal com o `.venv` ativo:

```bash
tensorboard --logdir outputs/training/runs --port 6006
```

Abra http://localhost:6006 enquanto o treino roda.


In [ ]:
logdir = repo_dir / 'outputs' / 'training' / 'runs'
logdir.mkdir(parents=True, exist_ok=True)
print('Log dir:', logdir)
print('Terminal:')
print(f'  tensorboard --logdir {logdir} --port 6006')


## Treinar (LoRA local)

Acompanhe `outputs/training/live.json` e o CSV enquanto roda.


In [ ]:
config_path = repo_dir / 'configs' / 'train_local.yaml'
output_dir = repo_dir / 'models' / 'qwen3-legacy-doc-lora-local'
metrics_csv = repo_dir / 'outputs' / 'training' / 'metrics.csv'
live_json = repo_dir / 'outputs' / 'training' / 'live.json'

cmd = [
    sys.executable, 'scripts/train_lora.py',
    '--config', str(config_path),
    '--backend', BACKEND,
    '--output-dir', str(output_dir),
]
print('Comando:', ' '.join(cmd))
print('Monitor ao vivo:', live_json)
print('CSV:', metrics_csv)

proc = subprocess.Popen(cmd, cwd=repo_dir)
try:
    while proc.poll() is None:
        if live_json.exists():
            print(live_json.read_text(encoding='utf-8'))
            print('-' * 40)
        time.sleep(20)
except KeyboardInterrupt:
    proc.terminate()
    raise
rc = proc.wait()
assert rc == 0, f'Treino falhou com código {rc}'
print('Treino concluído.')


## Curva de loss


In [ ]:
import pandas as pd

assert metrics_csv.exists(), f'Sem métricas em {metrics_csv}'
df = pd.read_csv(metrics_csv)
print(df.tail(20).to_string(index=False))

try:
    import matplotlib.pyplot as plt
    plot_df = df.dropna(subset=['loss'])
    if not plot_df.empty:
        plt.figure(figsize=(8, 4))
        plt.plot(plot_df['step'], plot_df['loss'], label='train loss')
        eval_df = df.dropna(subset=['eval_loss'])
        if not eval_df.empty:
            plt.plot(eval_df['step'], eval_df['eval_loss'], label='eval loss')
        plt.xlabel('step')
        plt.ylabel('loss')
        plt.title('Treino local — loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
except Exception as exc:
    print('Plot opcional indisponível:', exc)


## Conferir adapter


In [ ]:
import json
assert (output_dir / 'adapter_config.json').exists(), 'Adapter não salvo'
print('Arquivos:')
for path in sorted(output_dir.iterdir()):
    print('-', path.name)
metrics_path = output_dir / 'metrics.json'
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2)[:1000])
print('Adapter em', output_dir)
